# Chương 1 – Vì sao cần Model Adaptation? (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter01/chapter01_colab.ipynb)

Notebook này tái hiện sidebar §1.6 *"What the gap actually looks like"*: cùng một câu hỏi về một
chính sách hư cấu được gửi tới (1) model gốc Qwen3-4B, (2) adapter LoRA của chương 5 và
(3) model SFT của chương 6. Cuối notebook có phần **so sánh thêm với Claude** qua API key của bạn.

**GPU khuyến nghị:** L4 (24 GB) hoặc A100. T4 có thể chạy được phần inference nhưng chậm.

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

In [ ]:
import os, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab
# Sách chỉ dùng PyTorch. Không cho transformers nạp TensorFlow có sẵn trên Colab
# (tránh cảnh báo giả "Could not find cuda drivers" của TensorFlow).
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `ANTHROPIC_API_KEY` = API key Claude của bạn (**bắt buộc** cho notebook này)
- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [ ]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY", required=True)
hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

### Tải model đã train sẵn từ Hugging Face

Tác giả sách đã publish model của từng chương tại [`bahree/ModelAdaptationBook`](https://huggingface.co/bahree/ModelAdaptationBook). Ô dưới tải đúng các thư mục cần cho chương này về `/content/hub/` để các script dùng như đường dẫn local (không cần tự train các chương trước).

In [ ]:
from huggingface_hub import snapshot_download

HUB_DIR = "/content/hub"
SUBFOLDERS = ["ch5-lora", "ch6-sft"]
snapshot_download("bahree/ModelAdaptationBook", local_dir=HUB_DIR,
                  allow_patterns=[f"{s}/*" for s in SUBFOLDERS])
for s in SUBFOLDERS:
    print(s, "->", f"{HUB_DIR}/{s}", sorted(os.listdir(f"{HUB_DIR}/{s}"))[:6])

## 1. Chỉ chạy model gốc (base-only)

Nhanh nhất, không cần artifact nào khác.

In [ ]:
run("python -m chapter01.run_sidebar_example --base_only")

## 2. Chạy cả ba cấu hình (base, LoRA chương 5, SFT chương 6)

Dùng các model đã tải từ Hugging Face ở bước 0.

In [ ]:
run("python -m chapter01.run_sidebar_example "
    "--lora_dir /content/hub/ch5-lora "
    "--sft_dir /content/hub/ch6-sft")

In [ ]:
import json, pathlib
p = pathlib.Path("chapter01/sidebar_outputs.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:4000] if p.exists() else f"Chưa có {p}")

## 3. (Bonus) So sánh với Claude

Gửi cùng system prompt + câu hỏi tới Claude để thấy cách một model frontier xử lý câu hỏi về chính sách mà nó không có thông tin.

In [ ]:
import anthropic
from chapter01.run_sidebar_example import PROMPT, SYSTEM

client = anthropic.Anthropic()
resp = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=2000,
    system=SYSTEM,
    messages=[{"role": "user", "content": PROMPT}],
)
print("PROMPT:", PROMPT, "\n")
print("".join(b.text for b in resp.content if b.type == "text"))